# Label choice in a care-management algorithm

## Section 5 — Rebuild and compare the ranking

We will train one LASSO model to predict `gagne_sum_t`, the recorded active
chronic-condition count, then compare its top-3% allocation with the commercial
algorithm's top 3% on the same held-out patient-year rows.

Race is excluded from model training. It is retained only to audit who each
rule selects and how observed outcomes compare across groups.


## START/RESTART HERE

Before Task 1, run every setup code cell below in order. Each setup cell is
labeled `# RUN THIS CELL FIRST`; continue until you reach **Task 1**.

The setup imports packages, loads the data, fixes a 50/50 split and seed, and
defines the short classroom penalty grid. The full nine-value reference grid
remains visible as a comment for transparency but is not run in class.


In [ ]:
# RUN THIS CELL FIRST
from pathlib import Path

import numpy as np
import pandas as pd


In [ ]:
# RUN THIS CELL FIRST
DATA_COMMIT = "daceb25bba00e65d7b05882f049e229a8bedb60c"
DATA_URL = (
    "https://gitlab.com/labsysmed/dissecting-bias/-/raw/"
    f"{DATA_COMMIT}/data/data_new.csv"
)
LOCAL_DATA_PATH = Path("data_new.csv")


def load_data(required_columns):
    """Load the course data from a local copy or the pinned public URL."""
    source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.is_file() else DATA_URL
    data = pd.read_csv(source)

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return data


In [ ]:
# RUN THIS CELL FIRST
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
# RUN THIS CELL FIRST
RANDOM_STATE = 707
TEST_SIZE = 0.5

# Full reference grid used during instructor validation:
# RELATIVE_ALPHA_GRID = np.logspace(-4, 0, 9)

# The active classroom grid keeps the validated full-grid winner and its
# immediate neighboring candidates while reducing the search to 15 fold-fits.
RELATIVE_ALPHA_GRID = np.array([1e-3, 10**-2.5, 1e-2])

required_columns = ["race", "risk_score_t", "gagne_sum_t"]
df = load_data(required_columns)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


## Task 1 — Build leakage-safe modeling data

### 1A — Choose the predictors

Create `feature_columns`, a Python list in dataset order containing:

- every prior-year variable whose name ends in `_tm1`; and
- every demographic variable whose name starts with `dem_` and does not
  contain `race`.

Display the length of the list. The verified data should produce 149 predictor
names.


In [ ]:
# TODO


### 1B — Make the train-test split

Use one `train_test_split` call on the source-row index with
`test_size=TEST_SIZE` and `random_state=RANDOM_STATE`. Store the resulting NumPy
arrays as `train_index` and `test_index`.

Then create:

- `X_train` and `X_test`: predictor DataFrames using `feature_columns`; and
- `y_train` and `y_test`: matching `gagne_sum_t` Series.

Display the four shapes. The split should produce 24,392 rows in each half,
with 149 predictor columns.


In [ ]:
# TODO


### 1C — Check the feature names

Print two direct Boolean checks that confirm:

1. no name in `feature_columns` contains `race`; and
2. no name in `feature_columns` ends in current-year `_t`.

Both checks should display `True`.


In [ ]:
# TODO


## Task 2 — Train and evaluate one illness-burden LASSO

### 2A — Fit the classroom-sized search

Complete the supplied template. The `alpha_grid` line is already filled in for
class; replace each `...` in the remaining lines to create:

- `alpha_grid`: the three-value classroom grid for the LASSO penalty, scaled by
  the population standard deviation of `y_train`;
- `lasso_pipeline`: a Pipeline whose `scale` step is `StandardScaler()` and
  whose `model` step is `Lasso(alpha=1.0, max_iter=200_000, tol=0.001)`;
- `cv`: `KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)`;
- `illness_search`: a `GridSearchCV` that tunes `model__alpha`, scores with
  negative root mean squared error, uses `cv`, and sets `n_jobs=1`.

The short grid has only three values because the exhaustive search is too slow
for class and instructor validation has already confirmed that one performs
well. Fit `illness_search` with `X_train` and `y_train`. This performs 15
fold-fits plus one refit. Do not summarize `cv_results_`.


In [ ]:
# TODO

alpha_grid = RELATIVE_ALPHA_GRID * y_train.std(ddof=0)  # USE THIS ALPHA GRID IN CLASS

lasso_pipeline = Pipeline(...)

cv = KFold(...)

illness_search = GridSearchCV(
    lasso_pipeline,
    param_grid=...,
    scoring=...,
    cv=...,
    n_jobs=...,
)

illness_search.fit(..., ...);


### 2B — Evaluate on the untouched test half

Create:

- `illness_predictions`: a 24,392-element NumPy array from the selected LASSO;
- `mean_baseline_predictions`: a 24,392-element array that repeats only the
  training-set mean of `gagne_sum_t`;
- `baseline_rmse`: the held-out RMSE of the training-mean predictions; and
- `lasso_rmse`: the held-out RMSE of the LASSO predictions.

Use the supplied `rmse` helper and print the two RMSE values rounded to three
decimal places. Do not report the selected alpha or construct a DataFrame just
to display the results. Does LASSO improve on the baseline algorithm that gives
every patient the same training-set mean prediction?


In [ ]:
# TODO


**Your interpretation:** [State whether LASSO improves on the held-out mean
baseline and identify the units of both RMSE values.]


## Task 3 — Compare two fixed-capacity decisions with a random benchmark

Both policies treat the top 3% of the same 24,392 held-out rows, so each has
exactly 732 places. Run the single function below without editing it. This is
the deterministic top-fraction rule used in Section 3: larger scores receive
higher priority, and source-row order resolves any tie at the boundary.


In [ ]:
def select_exact_top_fraction(scores, fraction=0.03):
    """Select the highest-scoring fraction of rows.

    Parameters
    ----------
    scores : pandas.Series
        Scores indexed by source-row identifier. Larger values receive higher
        priority.
    fraction : float, default=0.03
        Fraction of rows to select.

    Returns
    -------
    pandas.Series
        Boolean Series aligned with ``scores``, where ``True`` identifies a
        selected row.

    Notes
    -----
    The function selects ``ceil(fraction * n)`` rows. Ties at the selection
    boundary are resolved by ascending source-row index.
    """
    number_selected = int(np.ceil(fraction * len(scores)))
    ranking = pd.DataFrame(
        {"score": scores, "source_row": scores.index},
        index=scores.index,
    )
    selected_index = (
        ranking.sort_values(
            ["score", "source_row"],
            ascending=[False, True],
            kind="mergesort",
        )
        .head(number_selected)
        .index
    )

    selected = pd.Series(False, index=scores.index, name="selected")
    selected.loc[selected_index] = True
    return selected


### 3A — Apply the LASSO-based policy

Create `test_results`, a DataFrame indexed by original source row. Begin with
`race`, `risk_score_t`, and observed `gagne_sum_t` for the held-out rows. Add:

- `lasso_prediction`: an explicitly indexed Series made from
  `illness_predictions`; and
- `lasso_selected`: the Boolean Series returned by
  `select_exact_top_fraction` when applied to `lasso_prediction` with
  `fraction=0.03`.

Use the template to preserve row alignment, fill in the `...`, and print the
number selected. It should be 732.


In [ ]:
# TODO

test_results = df.loc[
    test_index,
    ["race", "risk_score_t", "gagne_sum_t"],
].copy()

test_results["lasso_prediction"] = pd.Series(
    illness_predictions,
    index=...,
)

test_results["lasso_selected"] = select_exact_top_fraction(
    ...,
    fraction=...,
)

print("LASSO-selected rows:", test_results["lasso_selected"].sum())


### 3B — Apply the commercial-score policy

Apply `select_exact_top_fraction` to `risk_score_t` in `test_results`, again
using `fraction=0.03`. Store the resulting Boolean Series as
`commercial_selected` and print the number selected. It should also be 732.


In [ ]:
# TODO

test_results["commercial_selected"] = select_exact_top_fraction(
    ...,
    fraction=...,
)

print(
    "Commercial-score-selected rows:",
    test_results["commercial_selected"].sum(),
)


### 3C — Count treated people in each group

Create `lasso_treated` and `commercial_treated` by filtering `test_results` on
the corresponding selection indicator. Then create:

- `lasso_treated_counts`: a two-element Series with the numbers of Black and
  White patients treated by the LASSO policy; and
- `commercial_treated_counts`: the corresponding Series for the commercial
  policy.

Use `race_order` and `reindex` in the template so both Series display Black
first and White second. Print the two Series directly; do not construct a
reporting DataFrame.


In [ ]:
# TODO

race_order = ["black", "white"]

lasso_treated = test_results.loc[...]
commercial_treated = test_results.loc[...]

lasso_treated_counts = (
    lasso_treated["race"]
    .value_counts()
    .reindex(..., fill_value=0)
)
commercial_treated_counts = (
    commercial_treated["race"]
    .value_counts()
    .reindex(..., fill_value=0)
)

print("LASSO policy — treated people by race")
print(lasso_treated_counts)
print("\nCommercial policy — treated people by race")
print(commercial_treated_counts)


### 3D — Compare observed illness burden across policies

Add a naive benchmark that treats 3% of the held-out rows uniformly at random.
Create:

- `number_treated`: the exact capacity, 732 rows;
- `random_treated`: a uniformly sampled 732-row DataFrame from `test_results`,
  using `random_state=RANDOM_STATE` so the classroom result is reproducible;
- `random_mean_gagne_by_race`: the mean observed `gagne_sum_t` for randomly
  treated Black and White patients; and
- `random_gagne_gap`: the corresponding Black-minus-White mean difference.

Also create the corresponding mean Series and gaps for the two score-based
policies:

- `lasso_mean_gagne_by_race`; and
- `commercial_mean_gagne_by_race`.

Define every gap as the Black mean minus the White mean, so a positive value
means the Black treated group has the higher observed mean. Print the random,
commercial, and LASSO mean Series and gaps directly; do not construct a
reporting DataFrame. Do the commercial-score means suggest that it prioritizes
recorded illness burden better than uniform random selection?


In [ ]:
# TODO

number_treated = int(np.ceil(0.03 * len(test_results)))
random_treated = test_results.sample(
    n=...,
    random_state=...,
)

random_mean_gagne_by_race = (
    random_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(...)
)

lasso_mean_gagne_by_race = (
    lasso_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(...)
)
commercial_mean_gagne_by_race = (
    commercial_treated.groupby("race")["gagne_sum_t"]
    .mean()
    .reindex(...)
)

random_gagne_gap = ...
lasso_gagne_gap = ...
commercial_gagne_gap = ...

print("Random policy — mean gagne_sum_t by race")
print(random_mean_gagne_by_race.round(3))
print(f"Black - White gap: {random_gagne_gap:.3f}")

print("\nCommercial policy — mean gagne_sum_t by race")
print(commercial_mean_gagne_by_race.round(3))
print(f"Black - White gap: {commercial_gagne_gap:.3f}")

print("\nLASSO policy — mean gagne_sum_t by race")
print(lasso_mean_gagne_by_race.round(3))
print(f"Black - White gap: {lasso_gagne_gap:.3f}")


## END-OF-SECTION CHECKPOINT

Use the outputs from Tasks 3C and 3D to answer:

1. How does the number of Black and White patients treated change when moving
   from the commercial policy to the LASSO-based policy?
2. Does the commercial-score policy select patients with higher observed
   `gagne_sum_t` than uniform random selection within both racial groups?
3. Under the random, commercial, and LASSO policies, what is the direction and
   size of the Black-minus-White gap in mean observed `gagne_sum_t`?

**Your interpretation:** [Write two or three sentences here.]
